In [ ]:
import pandas as pd
from pathlib import Path

import rasterio
from rasterio.windows import Window

In [ ]:
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_ICEYE_ORG = DIR_DATA / "1-5_intermediate" / "1_ICEYE_aligned"
DIR_ICEYE_PROCESSED = DIR_DATA / "2_processed" / "1_ICEYE"
FILEPATH_PATCH_CENTERS = DIR_METADATA / "patch-centers.csv"
FILEPATH_ICEYE_LIST = DIR_METADATA / "iceye-list.csv"

PATCH_SIZE = 256  # pixels
PATCH_SIZE_HALF = PATCH_SIZE // 2


In [ ]:
flag_test = None  # 1: used for test data, otherwise not

patches = pd.read_csv(FILEPATH_PATCH_CENTERS)
images = pd.read_csv(FILEPATH_ICEYE_LIST)

for _, img in images.iterrows():
    # Skip images that are never used
    if img["count"] == 0:
        continue

    print(img["dirname"])

    dir_out = DIR_ICEYE_PROCESSED / img["dirname"]
    dir_out.mkdir(parents=True, exist_ok=True)
    image_path = DIR_ICEYE_ORG / img["filename"]
    image_path = DIR_ICEYE_ORG / (
        image_path.stem + "_EPSG2958_res05m.tif"
    )
    image_year = pd.to_datetime(img["date"]).year

    # Read test flag
    flag_test = img["test"]

    with rasterio.open(image_path) as src:

        print("====================================")
        print(image_path.name)
        print(f"Year      : {image_year}")
        print(f"CRS       : {src.crs}")
        print(f"Size      : {src.width} x {src.height}")
        print(f"Bounds    : {src.bounds}")
        print(f"Transform :\n{src.transform}")
        print(f"Resolution: {src.res}")
        print("====================================")

        transform = src.transform

        for _, patch in patches.iterrows():
            # Skip patches that don't belong to the current split
            if patch["usage"] == "test" and flag_test != 1:
                continue

            print(f"Patch: {patch['id']}, Year: {patch['year']}")

            if patch["year"] != image_year:
                print(f"Patch {patch['id']} is not from {image_year}")
                continue

            x = patch["x"]
            y = patch["y"]

            row, col = src.index(x, y)

            window = Window(
                col - PATCH_SIZE_HALF,
                row - PATCH_SIZE_HALF,
                PATCH_SIZE,
                PATCH_SIZE,
            )

            if (
                window.col_off < 0
                or window.row_off < 0
                or window.col_off + PATCH_SIZE > src.width
                or window.row_off + PATCH_SIZE > src.height
            ):
                print(f"Patch {patch['id']} is out of bounds")
                continue

            patch_img = src.read(window=window)

            profile = src.profile.copy()
            profile.update(
                width=PATCH_SIZE,
                height=PATCH_SIZE,
                transform=rasterio.windows.transform(window, transform),
                driver="GTiff",
                compress="LZW",
            )

            output_path = dir_out / f"{img['iceye_id']}_{patch['id']}.tif"

            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(patch_img)